# 01 — Exploratory Data Analysis

This notebook explores the **LTMM (Long Term Movement Monitoring)** dataset:
- Clinical/demographic metadata
- Lab-walk accelerometer signals (1-min, 6-channel, 100 Hz)
- Faller vs non-faller class distributions
- Signal visualisation for representative subjects

In [ ]:
import sys, pathlib
# Ensure the src directory is importable
ROOT = pathlib.Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from posturisk.preprocess import (
    load_clinical_data,
    load_lab_walk_features,
    parse_hea,
    read_wfdb_signal,
    DEFAULT_RAW_DIR,
    DEFAULT_PROCESSED_DIR,
)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
%matplotlib inline

## 1. Clinical / Demographic Data

In [ ]:
clinical = load_clinical_data()
print(f'Shape: {clinical.shape}')
print(f'Columns: {list(clinical.columns)}')
clinical.head(10)

In [ ]:
# Summary statistics
clinical.describe().round(2)

In [ ]:
# Missing values
missing = clinical.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot.barh(ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_title('Missing Values in Clinical Data')
    ax.set_xlabel('Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('No missing values!')

## 2. Class Distribution (Fallers vs Non-Fallers)

The clinical Excel file only contains CO (non-faller) data — faller labels come from the file prefix in the signal recordings.

In [ ]:
# Load the processed features to get the full dataset with labels
features_path = DEFAULT_PROCESSED_DIR / 'features.csv'
if features_path.exists():
    df = pd.read_csv(features_path)
else:
    # Run preprocessing if needed
    from posturisk.preprocess import run_pipeline
    df = run_pipeline()

print(f'Total subjects: {len(df)}')
print(f'Features: {len(df.columns)}')
print()
print(df['is_faller'].value_counts().rename({0: 'Non-Faller', 1: 'Faller'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['is_faller'].value_counts().sort_index()
labels = ['Non-Faller (CO)', 'Faller (FL)']
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.show()

ratio = counts.min() / counts.max()
print(f'Balance ratio: {ratio:.2f} (1.0 = perfectly balanced)')

## 3. Clinical Score Distributions by Faller Status

Note: Clinical scores are only available for CO (non-faller) subjects in this dataset.

In [ ]:
clinical_cols = ['age', 'tug', 'fsst', 'berg', 'dgi', 'abc_pct', 'mmse', 'moca']
available_cols = [c for c in clinical_cols if c in df.columns]

n = len(available_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(available_cols):
    ax = axes[i]
    for label, color, name in [(0, '#2ecc71', 'Non-Faller'), (1, '#e74c3c', 'Faller')]:
        subset = df[df['is_faller'] == label][col].dropna()
        if len(subset) > 0:
            ax.hist(subset, alpha=0.6, color=color, label=name, bins=12, edgecolor='white')
    ax.set_title(col)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Clinical Scores by Faller Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Raw Accelerometer Signals — Example Subjects

Comparing 1-minute lab walk recordings between a non-faller and a faller.

In [ ]:
lab_dir = DEFAULT_RAW_DIR / 'LabWalks'

def plot_subject_signals(hea_path, title_prefix=''):
    """Plot all 6 channels for a single subject."""
    header = parse_hea(hea_path)
    signals = read_wfdb_signal(hea_path.with_suffix('.dat'), header)
    t = np.arange(signals.shape[0]) / header.sample_rate
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
    channel_names = ['Vertical Acc (g)', 'Medio-Lateral Acc (g)', 'Antero-Posterior Acc (g)',
                     'Yaw Velocity (deg/s)', 'Pitch Velocity (deg/s)', 'Roll Velocity (deg/s)']
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
    
    for i, (ax, name, color) in enumerate(zip(axes.flatten(), channel_names, colors)):
        ax.plot(t, signals[:, i], color=color, linewidth=0.5, alpha=0.8)
        ax.set_ylabel(name, fontsize=9)
        ax.grid(alpha=0.3)
    
    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Time (s)')
    
    age_str = header.comments.get('age', '?')
    sex_str = header.comments.get('sex', '?')
    fig.suptitle(f'{title_prefix}{header.record_name} (Age: {age_str}, Sex: {sex_str})',
                 fontsize=14)
    plt.tight_layout()
    plt.show()

# Plot a non-faller
co_files = sorted(lab_dir.glob('co*_base.hea'))
fl_files = sorted(lab_dir.glob('fl*_base.hea'))

if co_files:
    print(f'Found {len(co_files)} non-faller lab walks')
    plot_subject_signals(co_files[0], title_prefix='NON-FALLER: ')

if fl_files:
    print(f'Found {len(fl_files)} faller lab walks')
    plot_subject_signals(fl_files[0], title_prefix='FALLER: ')

## 5. Signal Feature Distributions

Comparing key accelerometer features between fallers and non-fallers.

In [ ]:
signal_feat_cols = ['v_acc_rms', 'ml_acc_rms', 'ap_acc_rms',
                    'acc_magnitude_mean', 'acc_magnitude_std',
                    'v_acc_dom_freq', 'ml_acc_dom_freq',
                    'v_acc_spectral_entropy', 'ml_acc_spectral_entropy']
available_sig = [c for c in signal_feat_cols if c in df.columns]

n = len(available_sig)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(available_sig):
    ax = axes[i]
    non_faller = df[df['is_faller'] == 0][col].dropna()
    faller = df[df['is_faller'] == 1][col].dropna()
    ax.boxplot([non_faller, faller], labels=['Non-Faller', 'Faller'],
               patch_artist=True,
               boxprops=[dict(facecolor='#2ecc71', alpha=0.6),
                         dict(facecolor='#e74c3c', alpha=0.6)][0:1]*2)
    # Color the boxes individually
    bp = ax.boxplot([non_faller, faller], labels=['Non-Faller', 'Faller'],
                    patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('#e74c3c')
    bp['boxes'][1].set_alpha(0.6)
    ax.set_title(col)
    ax.grid(alpha=0.3, axis='y')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Signal Features: Fallers vs Non-Fallers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Correlation Heatmap

In [ ]:
# Select a subset of features for readability
heatmap_cols = ['is_faller', 'age', 'tug', 'fsst', 'berg', 'dgi',
                'v_acc_rms', 'ml_acc_rms', 'ap_acc_rms',
                'acc_magnitude_mean', 'v_acc_dom_freq',
                'v_acc_spectral_entropy', 'duration_s']
heatmap_cols = [c for c in heatmap_cols if c in df.columns]

corr = df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(heatmap_cols)))
ax.set_yticks(range(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(heatmap_cols, fontsize=9)

# Annotate cells
for i in range(len(heatmap_cols)):
    for j in range(len(heatmap_cols)):
        val = corr.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Summary

Key findings:
- **73 subjects** total: 38 non-fallers (CO), 35 fallers (FL)
- **Near-balanced** classes (~48% fallers)
- Clinical metadata only available for non-faller group
- **6-channel signals** at 100 Hz: 3 accelerometer + 3 gyroscope
- **89 features** extracted per subject (17 clinical + 69 signal + 3 metadata)
- Signal features (RMS, dominant frequency, spectral entropy) show potential discriminative power